# Drought-Impact Prediction Benchmark — end-to-end demo

This notebook walks the full benchmark pipeline on **synthetic data** — no flashdry
predictors and no network access to USDA RMA required. It exercises exactly the code
that runs on the real data locally: RMA parse → labels → predictor aggregation →
coverage → assemble → splits → baselines.

Install first: `pip install -e benchmarks/drought_impact`


## 1. Build a tiny synthetic dataset

Four counties × three years. 2012 is the extreme drought year; county `17003` in 2010
has liability but no drought row (a true-negative zero-loss county-year).


In [ ]:
import pandas as pd
from drought_impact.config import BenchmarkConfig
from drought_impact.rma import COL_COLUMNS

COUNTIES = {  # GEOID -> (state, county, abbrev, lat, lon)
    '17001': ('17','001','IL',39.9,-90.6), '17003': ('17','003','IL',37.2,-89.4),
    '19001': ('19','001','IA',41.0,-94.4), '19003': ('19','003','IA',40.7,-95.0),
}
YEARS = [2010, 2011, 2012]
DROUGHT = {2010: 10_000.0, 2011: 50_000.0, 2012: 300_000.0}  # loss_cost = D / 1e6

def col_row(year, geoid, cause, liability, indemnity, acres):
    s,c,ab,_,_ = COUNTIES[geoid]
    v = dict(commodity_year=str(year), state_code=s, state_abbrev=ab, county_code=c,
             county_name=f'COUNTY_{c}', commodity_code='0041', commodity_name='CORN',
             insurance_plan_code='01', insurance_plan_abbrev='APH', coverage_category='A',
             stage_code='H', cause_of_loss_code='10' if cause=='Drought' else '11',
             cause_of_loss_description=cause, month_of_loss='7', month_of_loss_name='JUL',
             year_of_loss=str(year), policies_earning_premium='50', policies_indemnified='10',
             net_planted_quantity=f'{acres:.1f}', net_endorsed_acres='0.0',
             liability=f'{liability:.1f}', total_premium=f'{liability*0.08:.1f}',
             producer_paid_premium='0.0', subsidy='0.0', state_private_subsidy='0.0',
             additional_subsidy='0.0', efa_premium_discount='0.0', net_determined_quantity='0.0',
             indemnity_amount=f'{indemnity:.1f}', loss_ratio='0.0')
    return [v[k] for k in COL_COLUMNS]

In [ ]:
from pathlib import Path
work = Path('demo_work'); raw = work/'raw'; raw.mkdir(parents=True, exist_ok=True)
for year in YEARS:
    rows = []
    for g in COUNTIES:
        rows.append(col_row(year, g, 'Hail', 1_000_000.0, 5_000.0, 8_000.0))
        if not (g=='17003' and year==2010):
            rows.append(col_row(year, g, 'Drought', 0.0, DROUGHT[year], 0.0))
    (raw/f'colsom_{year}.txt').write_text('\n'.join('|'.join(r) for r in rows)+'\n', encoding='latin-1')

# Synthetic flashdry feature_table: weekly _anom features inside the season window.
recs = []
for g,(_,_,_,lat,lon) in COUNTIES.items():
    for year in YEARS:
        for wk,doy in enumerate(range(70,210,18)):
            date = pd.Timestamp(year,1,1)+pd.Timedelta(days=doy-1)
            spei = -2.5 if year==2012 else (-0.3 if year==2011 else 0.1)
            recs.append(dict(GEOID=g, date=date, year=year, lat=lat, lon=lon,
                             spei_anom=spei+0.05*wk, ndvi_anom=(-1.8 if year==2012 else 0.2)-0.02*wk))
ft_path = work/'feature_table.parquet'; pd.DataFrame(recs).to_parquet(ft_path, index=False)
nass = pd.DataFrame([dict(GEOID=g, year=y, planted_acres=8000/0.8) for g in COUNTIES for y in YEARS])
nass_path = work/'nass.parquet'; nass.to_parquet(nass_path, index=False)
print('fixtures written to', work)

## 2. Configure and assemble the benchmark


In [ ]:
cfg = BenchmarkConfig(states=['17','19'], year_min=2010, year_max=2012, crop='CORN',
                      cutoff_doy=212, binary_threshold=0.10, spatial_blocks_side=2,
                      temporal_test_years=[2012], feature_table_path=ft_path,
                      nass_acres_path=nass_path, rma_data_dir=raw, output_dir=work/'processed')
cfg.validate_all()

from drought_impact.assemble import assemble_benchmark
benchmark, manifest = assemble_benchmark(cfg, write=True)
print('build_fingerprint:', manifest['build_fingerprint'])
benchmark[['GEOID','year','drought_loss_cost','significant_loss','insured_acre_fraction']]

2012 shows the largest loss-cost, and county `17003` in 2010 is a zero-loss true
negative — exactly the invariants the real dataset must satisfy.


## 3. Splits and baselines


In [ ]:
from drought_impact.predictors import extract_centroids
from drought_impact.splits import build_splits, validate_splits
from drought_impact.baselines import run_baselines

centroids = extract_centroids(pd.read_parquet(ft_path))
splits = build_splits(benchmark, centroids, cfg); validate_splits(splits)
print('temporal test rows:', splits['temporal']['test'])
print('spatial-block folds:', len(splits['spatial_block']['folds']))

leaderboard = run_baselines(benchmark, splits)
leaderboard

On the real RMA + flashdry data, this leaderboard is where **severity ≠ impact** shows
up: a strong drought *severity* index (WxCond) is informative but imperfect on the
drought *loss* target, and ML baselines on the `_anom` predictors beat the naive rate.
